<a href="https://colab.research.google.com/github/Raymondycp/QuantFinance_AlgoTradingStrategy/blob/main/%5BQuantFinance%5DBacktest_Backtrader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install backtrader pandas yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 4.7 MB/s eta 0:00:00


In [ ]:
import backtrader as bt
import backtrader.indicators as btind
import backtrader.analyzers as btanalyzers
import backtrader.feeds as btfeeds
import datetime
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
class ProfessionalDailyStrategy(bt.Strategy):
    params = (
        ('trend_ema_period', 50),       # Period for trend EMA
        ('entry_ema_period', 20),       # Period for entry signal EMA
        ('exit_ema_period', 7),         # Period for exit signal EMA
        ('atr_period', 14),             # Period for ATR calculation
        ('risk_per_trade', 0.01),       # Risk percentage per trade
        ('rr_ratio', 1),                # Risk-reward ratio
        ('max_position_size', 0.1),     # Maximum position size as % of portfolio
        ('adx_threshold', 25),          # ADX threshold for strong trend
        ('volume_ma_period', 20),       # Period for volume moving average
        ('volume_threshold', 1.5),      # Volume threshold multiplier
        ('trail_percent', 0.05),       # Trailing stop percentage
    )

    def __init__(self):
        # Initialize indicators
        self.trend_ema = btind.EMA(period=self.p.trend_ema_period)
        self.entry_ema = btind.EMA(period=self.p.entry_ema_period)
        self.exit_ema = btind.EMA(period=self.p.exit_ema_period)
        self.adx = btind.ADX(period=14)
        self.rsi = btind.RSI(period=14)
        self.atr = btind.ATR(period=self.p.atr_period)
        self.volume_ma = btind.SMA(self.data.volume, period=self.p.volume_ma_period)

        # Initialize trading variables
        self.order = None              # Current order
        self.stop_price = None         # Stop loss price
        self.target_price = None       # Take profit price
        self.trade_size = None         # Size of trade
        self.trade_count = 0           # Total trade count
        self.winning_trades = 0        # Count of winning trades
        self.trail_stop = None         # Trailing stop price

    def next(self):
        # Return if there's a pending order
        if self.order:
            return

        # Wait until we have enough data for all indicators
        if len(self) < max(self.p.trend_ema_period, self.p.atr_period, self.p.volume_ma_period):
            return

        portfolio_value = self.broker.getvalue()
        current_price = self.data.close[0]

        # Calculate position size based on risk
        risk_amount = portfolio_value * self.p.risk_per_trade
        dollar_risk = self.atr[0] * 2
        self.trade_size = min(
            risk_amount / dollar_risk if dollar_risk > 0 else 0,
            (portfolio_value * self.p.max_position_size) / current_price
        )

        # Determine market conditions
        strong_trend = self.adx[0] > self.p.adx_threshold
        above_trend_ema = current_price > self.trend_ema[0]
        below_trend_ema = current_price < self.trend_ema[0]
        high_volume = self.data.volume[0] > (self.volume_ma[0] * self.p.volume_threshold)

        # Entry logic when not in position
        if not self.position:
            # Long entry conditions
            if (above_trend_ema and
                self.entry_ema[0] > self.exit_ema[0] and
                self.entry_ema[-1] <= self.exit_ema[-1] and
                (high_volume or strong_trend)):

                self.stop_price = current_price - 2 * self.atr[0]
                self.target_price = current_price + (2 * self.atr[0] * self.p.rr_ratio)
                self.order = self.buy(size=self.trade_size)

            # Short entry conditions
            elif (below_trend_ema and
                  self.entry_ema[0] < self.exit_ema[0] and
                  self.entry_ema[-1] >= self.exit_ema[-1] and
                  (high_volume or strong_trend)):

                self.stop_price = current_price + 2 * self.atr[0]
                self.target_price = current_price - (2 * self.atr[0] * self.p.rr_ratio)
                self.order = self.sell(size=self.trade_size)

        # Exit logic when in position
        else:
            if self.position.size > 0:  # Long position
                self.trail_stop = current_price * (1 - self.p.trail_percent)
                if current_price >= self.target_price or current_price <= max(self.stop_price, self.trail_stop):
                    self.order = self.close()

            elif self.position.size < 0:  # Short position
                self.trail_stop = current_price * (1 + self.p.trail_percent)
                if current_price <= self.target_price or current_price >= min(self.stop_price, self.trail_stop):
                    self.order = self.close()

    def notify_order(self, order):
        # Update trade statistics when order is completed
        if order.status in [order.Completed]:
            self.trade_count += 1
            if order.executed.pnl > 0:
                self.winning_trades += 1
            self.order = None

    def stop(self):
        # Print backtest results when strategy stops
        win_rate = (self.winning_trades / self.trade_count) * 100 if self.trade_count > 0 else 0
        print(f'Backtest completed. Total trades: {self.trade_count}, Win rate: {win_rate:.2f}%')
        print(f'Final portfolio value: {self.broker.getvalue():.2f}')

In [ ]:
# Download data
data = yf.download('AAPL', start='2015-01-01', end='2020-12-31', auto_adjust=False)

# Fix column names: Remove multi-index, keep only the first level
data.columns = data.columns.droplevel(1)  # Remove the 'AAPL' level
print(data.head())  # Check the modified column names

# Alternative more explicit renaming approach (if the above method doesn't work):
# data = data.rename(columns={
#     'Adj Close': 'AdjClose',
#     'Close': 'Close',
#     'High': 'High',
#     'Low': 'Low',
#     'Open': 'Open',
#     'Volume': 'Volume'
# })

[*********************100%***********************]  1 of 1 completed

Price       Adj Close      Close       High        Low       Open     Volume
Date                                                                        
2020-01-02  72.716072  75.087502  75.150002  73.797501  74.059998  135480400
2020-01-03  72.009140  74.357498  75.144997  74.125000  74.287498  146322800
2020-01-06  72.582916  74.949997  74.989998  73.187500  73.447502  118387200
2020-01-07  72.241531  74.597504  75.224998  74.370003  74.959999  108872000
2020-01-08  73.403656  75.797501  76.110001  74.290001  74.290001  132079200


In [ ]:
# Initialize Cerebro engine
cerebro = bt.Cerebro()
# Set initial cash amount
cerebro.broker.setcash(100000.0)
# Set commission rate (0.1%)
cerebro.broker.setcommission(commission=0.001)

# Add data feed
datafeed = btfeeds.PandasData(
    dataname=data,
    datetime=None,  # Use default index as datetime
    open='Open',    # Column name for open prices
    high='High',    # Column name for high prices
    low='Low',      # Column name for low prices
    close='Close',  # Column name for close prices
    volume='Volume',# Column name for volume
    openinterest=-1 # No open interest data
)
cerebro.adddata(datafeed)

# Add trading strategy
cerebro.addstrategy(ProfessionalDailyStrategy)

# Add performance analyzers
cerebro.addanalyzer(btanalyzers.SharpeRatio, _name='sharpe')      # Sharpe ratio analyzer
cerebro.addanalyzer(btanalyzers.DrawDown, _name='drawdown')       # Drawdown analyzer
cerebro.addanalyzer(btanalyzers.Returns, _name='returns')        # Returns analyzer
cerebro.addanalyzer(btanalyzers.TradeAnalyzer, _name='trades')   # Trade statistics analyzer

In [ ]:
# Print initial portfolio value
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

# Run backtest
results = cerebro.run()

# Print final portfolio value
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

# Print performance metrics
strat = results[0]
print('Sharpe Ratio:', strat.analyzers.sharpe.get_analysis()['sharperatio'])
print('Max Drawdown:', strat.analyzers.drawdown.get_analysis()['max']['drawdown'])
print('Annualized Return:', strat.analyzers.returns.get_analysis()['rnorm100'])

Starting Portfolio Value: 100000.00
Backtest completed. Total trades: 10, Win rate: 30.00%
Final portfolio value: 101043.07
Final Portfolio Value: 101043.07
Sharpe Ratio: -1.3772911758027575
Max Drawdown: 1.3655245892947807
Annualized Return: 0.2082453235876751
